# Gayanara E-commerce: Data Cleaning
## 0. Load dan Cek Awal Data

In [10]:
import pandas as pd
products = pd.read_csv("data/products.csv")
customers = pd.read_csv("data/customers.csv")
orders = pd.read_csv("data/orders.csv")
items = pd.read_csv("data/order_items.csv")
reviews = pd.read_csv("data/reviews.csv")
tables = {"products": products, "customers": customers, "orders": orders, "items": items, "reviews": reviews}
[print(n, d.shape, "total null:", d.isna().sum().sum(), "baris duplikat:", d.duplicated().sum()) for n, d in tables.items()]

products (300, 9) total null: 9 baris duplikat: 0
customers (800, 9) total null: 40 baris duplikat: 0
orders (3000, 12) total null: 3572 baris duplikat: 0
items (4986, 6) total null: 0 baris duplikat: 0
reviews (1500, 8) total null: 120 baris duplikat: 0


[None, None, None, None, None]

## 1. Standarisasi Kategori Produk

In [11]:
map_cat = {"dress":"Dress","jacket":"Jacket","jaket":"Jacket","pants":"Pants","celana":"Pants","accessories":"Accessories","aksesoris":"Accessories","kaos":"T-Shirt","t-shirt":"T-Shirt","kemeja":"Shirt","shirt":"Shirt"}
products["category_clean"] = products["category"].str.lower().map(map_cat)
print(products[products["category_clean"].isna()]["category"].unique())
print("gagal ke-map:", products["category_clean"].isna().sum())

[]
gagal ke-map: 0


In [12]:
print(products["category_clean"].value_counts())

category_clean
Jacket         56
Accessories    52
Pants          52
Dress          48
Shirt          48
T-Shirt        44
Name: count, dtype: int64


## 2. Email yang Dipakai Bersama

In [13]:
dup_email = customers[customers["email"].duplicated(keep=False)].sort_values("email")
dup_email[["customer_id", "name", "email", "city", "registration_date"]]

,customer_id,name,email,city,registration_date
44,CUST0045,Bagus Permata,baguspermata248@gmail.com,Bandung,2022-08-26
320,CUST0321,Teguh Iskandar,baguspermata248@gmail.com,Makassar,2022-04-26
699,CUST0700,Rina Lestari,rinalestari281@outlook.com,Banjarmasin,2023-03-31
754,CUST0755,Kirana Tarigan,rinalestari281@outlook.com,Tangerang,2023-04-22
187,CUST0188,Wulan Suryadi,wulansuryadi729@yahoo.com,Padang,2021-06-04
566,CUST0567,Olivia Lestari,wulansuryadi729@yahoo.com,Pekanbaru,2024-09-01


In [14]:
customers["email_shared"] = customers["email"].duplicated(keep=False)
print("customer dengan email dipakai bersama:", customers["email_shared"].sum())

customer dengan email dipakai bersama: 6


## 3. Order Sebelum Tanggal Registrasi

In [15]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
customers["registration_date"] = pd.to_datetime(customers["registration_date"])
reg_map = customers.set_index("customer_id")["registration_date"]
orders["order_before_reg"] = orders["order_date"] < orders["customer_id"].map(reg_map)
print(orders["order_before_reg"].sum(), "dari", len(orders), "order")

795 dari 3000 order


## 4. Validasi Total Order dan Definisi Revenue

In [16]:
s = items.groupby("order_id")["subtotal_idr"].sum().rename("items_sum")
chk2 = orders.merge(s, on="order_id")
chk2["expected"] = chk2["items_sum"] - chk2["discount_amount_idr"].fillna(0)
print((chk2["expected"] == chk2["total_amount_idr"]).sum(), "dari", len(chk2))

3000 dari 3000


In [17]:
orders["is_valid_revenue"] = orders["order_status"].isin(["delivered", "shipped", "processing"])
print(orders["is_valid_revenue"].value_counts())

is_valid_revenue
True     2521
False     479
Name: count, dtype: int64


## 5. Simpan Data Bersih

In [18]:
import os
os.makedirs("data_clean", exist_ok=True)
products.to_csv("data_clean/products_clean.csv", index=False)
customers.to_csv("data_clean/customers_clean.csv", index=False)
orders.to_csv("data_clean/orders_clean.csv", index=False)
items.to_csv("data_clean/order_items.csv", index=False)
reviews.to_csv("data_clean/reviews.csv", index=False)